In [36]:
import os
from dotenv import load_dotenv, find_dotenv
from google.cloud import compute_v1
from google.api_core.exceptions import NotFound
from pathlib import Path
import time
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError
import pandas as pd
from faker import Faker

In [37]:
load_dotenv(find_dotenv(usecwd=True), override=True)

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
ZONE = os.getenv("GCP_ZONE_REGION", "us-west1-b")
INSTANCE_NAME = os.getenv("GCP_INSTANCE_NAME", "ssm-console-compute")
MACHINE_TYPE = os.getenv("GCP_MACHINE_TYPE", "e2-medium")
IMAGE_PROJECT = os.getenv("GCP_SOURCE_IMAGE_PROJECT", "ubuntu-os-cloud")
IMAGE_FAMILY = os.getenv("GCP_SOURCE_IMAGE_FAMILY", "ubuntu-2404-lts-amd64")
print("Imagen:", IMAGE_PROJECT, IMAGE_FAMILY)  # verificación rápida
NETWORK_NAME = os.getenv("GCP_NETWORK_NAME", "global/networks/default")

PG_USER = os.getenv("PG_USER", "admin_user")
PG_PASSWORD = os.getenv("PG_PASSWORD", "SuperSecurePassword123!")
PG_DB = os.getenv("PG_DB", "smartdata_db")

Imagen: ubuntu-os-cloud ubuntu-2404-lts-amd64


In [38]:
SCRIPT_PATH = Path("/workspace/tarea1/shellScripts/startup_script.sh")
startup_script = SCRIPT_PATH.read_text(encoding="utf-8").replace("\r\n", "\n")
METADATA = {
    "startup-script": startup_script,
    "pg-user": PG_USER,
    "pg-password": PG_PASSWORD,
    "pg-db": PG_DB,
}


In [39]:
# ── Clientes ───────────────────────────────────────────────────────────
instances_client = compute_v1.InstancesClient()

In [40]:
def wait_op(operation, timeout=300):
    """Espera una operación y lanza excepción si GCP reporta error."""
    operation.result(timeout=timeout)

In [41]:
def get_instance():
    """Devuelve la VM si existe, o None si no existe."""
    try:
        return instances_client.get(project=PROJECT_ID, zone=ZONE, instance=INSTANCE_NAME)
    except NotFound:
        return None

In [42]:
def build_instance():
    """Especificación completa de la VM para crearla desde cero."""
    instance = compute_v1.Instance()
    instance.name = INSTANCE_NAME
    instance.machine_type = f"zones/{ZONE}/machineTypes/{MACHINE_TYPE}"

    disk = compute_v1.AttachedDisk()
    disk.boot = True
    disk.auto_delete = True
    initialize_params = compute_v1.AttachedDiskInitializeParams()
    initialize_params.source_image = f"projects/{IMAGE_PROJECT}/global/images/family/{IMAGE_FAMILY}"
    initialize_params.disk_size_gb = 30
    disk.initialize_params = initialize_params
    instance.disks = [disk]

    network_interface = compute_v1.NetworkInterface()
    network_interface.network = NETWORK_NAME
    access_config = compute_v1.AccessConfig()
    access_config.name = "External NAT"
    access_config.type_ = compute_v1.AccessConfig.Type.ONE_TO_ONE_NAT.name
    network_interface.access_configs = [access_config]
    instance.network_interfaces = [network_interface]

    instance.metadata = compute_v1.Metadata()
    instance.metadata.items = [compute_v1.Items(key=k, value=v) for k, v in METADATA.items()]
    instance.tags = compute_v1.Tags()
    instance.tags.items = ["postgres-server", "http-server"]
    return instance

In [43]:
def update_metadata(inst):
    """Actualiza solo las claves de METADATA, conservando las demás. Devuelve True si hubo cambios."""
    metadata = inst.metadata  # conserva el fingerprint
    current = {i.key: i.value for i in metadata.items}

    if all(current.get(k) == v for k, v in METADATA.items()):
        print("= Metadatos sin cambios.")
        return False

    items = [i for i in metadata.items if i.key not in METADATA]
    items += [compute_v1.Items(key=k, value=v) for k, v in METADATA.items()]
    metadata.items = items

    op = instances_client.set_metadata(
        project=PROJECT_ID, zone=ZONE, instance=INSTANCE_NAME, metadata_resource=metadata
    )
    wait_op(op)
    print("✓ Metadatos actualizados")
    return True

In [44]:
# ── Crear o actualizar ─────────────────────────────────────────────────
inst = get_instance()

if inst is None:
    print(f"Desplegando VM '{INSTANCE_NAME}' en {ZONE}...")
    op = instances_client.insert(project=PROJECT_ID, zone=ZONE, instance_resource=build_instance())
    wait_op(op)
    print("✓ VM creada exitosamente.")
else:
    print(f"La VM '{INSTANCE_NAME}' ya existe (estado: {inst.status}).")
    changed = update_metadata(inst)

    if inst.status == "TERMINATED":
        print("Encendiendo VM detenida...")
        wait_op(instances_client.start(project=PROJECT_ID, zone=ZONE, instance=INSTANCE_NAME))
        print("VM encendida (el startup script se ejecutará al arrancar).")
    elif changed:
        print("El nuevo startup script se aplicará en el próximo reinicio de la VM.")


La VM 'ssm-console-compute' ya existe (estado: RUNNING).
= Metadatos sin cambios.


In [45]:
# ── Datos de conexión ──────────────────────────────────────────────────
inst = get_instance()
changed = False
if inst is None:
    public_ip = inst.network_interfaces[0].access_configs[0].nat_i_p
    print(f"IP Pública: {public_ip}")
    print(f"DSN PostgreSQL: postgresql://{PG_USER}:{PG_PASSWORD}@{public_ip}:5432/{PG_DB}")

In [46]:
#Ejecucion de reinicio de la VM para que se ejecute el startup script
if inst is not None and inst.status == "RUNNING" and changed:
    print("Reiniciando VM para aplicar el startup script...")
    wait_op(instances_client.reset(project=PROJECT_ID, zone=ZONE, instance=INSTANCE_NAME))
else:
    print("Reinicio no necesario.")

Reinicio no necesario.


Probar Conexion desde Notebook a PostgresSQL de la VM creado en GCP

In [47]:
DSN = f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{public_ip}:5432/{PG_DB}"
engine = create_engine(DSN, pool_pre_ping=True, connect_args={"connect_timeout": 5})

for intento in range(30):  # hasta ~5 minutos
    try:
        with engine.connect() as conn:
            version = conn.execute(text("SELECT version();")).scalar()
            print("✓ Conectado:", version)
            break
    except OperationalError:
        print(f"Esperando a PostgreSQL... ({intento + 1}/30)")
        time.sleep(10)
else:
    print("✗ No se pudo conectar. Revisa el serial output de la VM.")

✓ Conectado: PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


Probar Base de Datos en Postgres creado en la VM de GCP

In [48]:
fake = Faker("es_MX")
df = pd.DataFrame([{
    "nombre": fake.name(),
    "email": fake.email(),
    "ciudad": fake.city(),
    "fecha_registro": fake.date_between("-2y", "today"),
} for _ in range(1000)])

df.to_sql("clientes", engine, if_exists="replace", index=False)

with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM clientes")).scalar())

1000


In [49]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM clientes LIMIT 5"))
    for row in result:
        print(row)

('Alberto Meraz Santacruz', 'guillermina86@example.org', 'Vieja Japón', datetime.date(2025, 4, 18))
('Lic. Adán Preciado', 'guillermo04@example.net', 'San Julio los bajos', datetime.date(2026, 4, 15))
('Adriana Linda Regalado', 'vmena@example.net', 'San Ana Luisa los altos', datetime.date(2025, 12, 9))
('Luz Verduzco', 'nancyecheverria@example.com', 'Nueva Namibia', datetime.date(2026, 8, 22))
('Sra. Virginia Mojica', 'jrocha@example.com', 'Nueva Botswana', datetime.date(2025, 4, 27))


In [50]:
pd.read_sql("SELECT * FROM clientes LIMIT 5", engine)

,nombre,email,ciudad,fecha_registro
0,Alberto Meraz Santacruz,guillermina86@example.org,Vieja Japón,2025-04-18
1,Lic. Adán Preciado,guillermo04@example.net,San Julio los bajos,2026-04-15
2,Adriana Linda Regalado,vmena@example.net,San Ana Luisa los altos,2025-12-09
3,Luz Verduzco,nancyecheverria@example.com,Nueva Namibia,2026-08-22
4,Sra. Virginia Mojica,jrocha@example.com,Nueva Botswana,2025-04-27


# Apagar VM al finalizar

In [51]:
instances_client.stop(project=PROJECT_ID, zone=ZONE, instance=INSTANCE_NAME)
# instances_client.start(...) para volver a encenderla

<google.cloud.compute_v1.services.instances.client.InstancesClient.stop.<locals>._CustomOperation at 0x75ba0ac423c0>